# ERA5 — eventi sullo score normalizzato per climatologia

Nel report annuale dello score STGAN originale le anomalie sopra la soglia globale erano soprattutto **mezzogiorni di aprile-maggio, ogni anno** (correlazione fra anni 0.975, lift mensile 0.36–2.2, orario 0.21–3.3): l'errore di predizione e' strutturalmente piu' alto in quel regime e il modello riceve solo giorno della settimana e ora, non il periodo dell'anno. Qui lo score diventa uno **z-score climatologico**:

`z = (score - media) / deviazione standard`, con media e deviazione calcolate sulla **stessa localita', stesso mese e stessa ora UTC negli altri anni** (leave-one-year-out: l'anno valutato non entra mai nella propria climatologia). Gruppi con meno di `--min-samples` valori (default 30) o varianza nulla restano NaN. La deviazione standard MC viene divisa per la stessa deviazione climatologica, quindi resta confrontabile con z. **Nessun nuovo training**: si parte dagli score gia' salvati.

Pipeline, come per lo score originale:
1. `scripts/run_era5_stgan.py climatology` legge la run STGAN e scrive una **run z** con lo stesso formato (`metadata.json`, `scores/anomaly_mean.npy`, `test_timestamps.npy`);
2. `scripts/run_era5_stgan.py events` su quella run crea cubo, cluster ed eventi, con le **stesse opzioni** della run eventi originale (soglia globale al 99esimo percentile degli z, morfologia, collegamenti);
3. visualizzazione: barre per timestamp, lista eventi, un timestamp, evoluzione di un evento;
4. controllo della stagionalita': report annuale dello z confrontato con quello dello score originale.

I passi 1 e 2 vengono saltati se le cartelle esistono gia' complete; per ricalcolare cancella `Z_RUN_DIR` o `EVENT_DIR`. Su griglia completa ciascun passo richiede alcuni minuti.

In [ ]:
from pathlib import Path
import sys, json, sqlite3, subprocess
from contextlib import closing
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'physiq_pv').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from physiq_pv.era5.seasonality import annual_report, seasonality_metrics
from physiq_pv.era5.visualization import (anomaly_activity, plot_anomaly_prevalence,
                                          plot_event_overview, plot_frame, plot_event, plot_event_snapshots)
RUN_DIR = ROOT / 'outputs/era5_cnn_mc_paper_run2'  # run STGAN (metadata.json + scores/) usata per gli eventi originali
ORIGINAL_EVENT_DIR = ROOT / 'outputs/era5_cnn_mc_paper_run2_events_top1'  # eventi sullo score originale
Z_RUN_DIR = ROOT / 'outputs/era5_cnn_mc_paper_run2_zclim'
EVENT_DIR = ROOT / 'outputs/era5_cnn_mc_paper_run2_zclim_events_top1'
MIN_SAMPLES = 30
PREVALENCE_TITLE = 'STGAN ERA5 z climatologico - percentuale sopra soglia a ogni timestamp'
SCRIPT = ROOT / 'scripts/run_era5_stgan.py'


def complete(directory):
    manifest = Path(directory) / 'metadata.json'
    return manifest.is_file() and json.loads(manifest.read_text(encoding='utf-8')).get('status') == 'complete'


def run_script(*arguments):
    command = [sys.executable, str(SCRIPT), *map(str, arguments)]
    print('>', ' '.join(command))
    result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True)
    if result.returncode:
        print(result.stdout[-4000:], result.stderr[-4000:], sep='\n')
        raise RuntimeError(f'Comando fallito ({result.returncode}); se la cartella di output e parziale, cancellala e riesegui')

## 1. Run z climatologica

Legge `anomaly_mean` e `anomaly_std` della run STGAN e scrive la run z. La tabella riporta anni e ore usati per la climatologia e quanti valori sono rimasti NaN.

In [ ]:
if complete(Z_RUN_DIR):
    print('Run z gia presente:', Z_RUN_DIR)
else:
    assert complete(RUN_DIR), f'RUN_DIR non trovato o incompleto: {RUN_DIR}'
    run_script('climatology', '--run-dir', RUN_DIR, '--output-dir', Z_RUN_DIR, '--min-samples', MIN_SAMPLES)
z_run = json.loads((Z_RUN_DIR / 'metadata.json').read_text(encoding='utf-8'))
normalization = z_run['normalization']
display(pd.Series({'metodo': normalization['method'], 'run sorgente': z_run['source_run_dir'],
                   'anni': f"{normalization['years'][0]}-{normalization['years'][-1]} ({len(normalization['years'])})",
                   'ore UTC': normalization['hours_utc'], 'min_samples': normalization['min_samples'],
                   'valori finiti': f"{normalization['finite_values']:,}", 'valori NaN': f"{normalization['nan_values']:,}"},
                  name='Run z'))

## 2. Eventi sulla run z

Stesse opzioni della run eventi originale (lette da `ORIGINAL_EVENT_DIR/metadata.json`), tranne l'eventuale soglia assoluta, che sullo z non avrebbe senso. Se la run originale non c'e', si usano i default di `events` (soglia globale al 99esimo percentile).

In [ ]:
EVENT_OPTIONS = ('top_percent', 'threshold_scope', 'kernel_size', 'opening_iterations', 'closing_iterations',
                 'min_cells', 'link_policy', 'min_overlap', 'dilation_cells', 'centroid_distance_km', 'chunk_size')
if complete(EVENT_DIR):
    print('Eventi z gia presenti:', EVENT_DIR)
else:
    options = []
    if complete(ORIGINAL_EVENT_DIR):
        original_config = json.loads((ORIGINAL_EVENT_DIR / 'metadata.json').read_text(encoding='utf-8'))['config']
        for name in EVENT_OPTIONS:
            if original_config.get(name) is not None:
                options += ['--' + name.replace('_', '-'), original_config[name]]
    run_script('events', '--run-dir', Z_RUN_DIR, '--output-dir', EVENT_DIR, *options)
metadata = json.loads((EVENT_DIR / 'metadata.json').read_text(encoding='utf-8'))
assert metadata['status'] == 'complete', 'Elaborazione eventi incompleta'
display(pd.Series(metadata['config'], name='Configurazione eventi z'))
print('Shape (T,H,W):', metadata['shape'])
print('Cluster:', metadata['n_clusters'], '| Eventi:', metadata['n_events'])

## Ogni timestamp di 3 ore, tutte le localita'

Una barra verticale per **ogni timestamp disponibile**, senza media o raggruppamento: l'altezza e' la percentuale di localita' con score **strettamente sopra** la soglia salvata nella tabella `frames` della run eventi, divisa per le localita' con score valido in quel timestamp. Con `threshold_scope=global` e `--top-percent 1` la soglia e' il 99esimo percentile di tutti gli score finiti, calcolato una sola volta: la quota varia nel tempo e non e' imposta all'1% in ogni frame. Il conteggio e' fatto **prima** di opening, closing e clustering; `activity.n_event_cells` conta invece le celle nei cluster finali. L'asse verticale e' una percentuale, non lo score. Imposta `VIEW_START` e `VIEW_END` per ingrandire un periodo mantenendo la risoluzione di 3 ore.

In [ ]:
VIEW_START = None  # es. '2025-01-01'; None mostra dall'inizio
VIEW_END = None    # es. '2025-06-01'; None mostra fino alla fine
activity = anomaly_activity(EVENT_DIR)
event_config = metadata['config']
if event_config.get('absolute_threshold') is not None:
    print(f"Soglia assoluta: {event_config['absolute_threshold']}")
elif event_config.get('threshold_scope', 'global') == 'global':
    print(f"Soglia globale: percentile {100 - event_config['top_percent']:g} = {activity.threshold.iloc[0]:.6g}")
else:
    print(f"Soglia percentile {100 - event_config['top_percent']:g} calcolata separatamente per ogni frame")
print(f'Celle sopra soglia: {activity.n_above_threshold.sum():,}/{activity.n_valid.sum():,} '
      f'({100 * activity.n_above_threshold.sum() / max(1, activity.n_valid.sum()):.3f}%); '
      f'timestamp con almeno una cella sopra soglia: {activity.n_above_threshold.gt(0).sum():,}/{len(activity):,}')
figure = plot_anomaly_prevalence(activity, VIEW_START, VIEW_END, title=PREVALENCE_TITLE)
display(figure)
plt.close(figure)

In [ ]:
with closing(sqlite3.connect(EVENT_DIR / 'events.sqlite')) as db:
    events = pd.read_sql_query('SELECT * FROM events ORDER BY max_score DESC LIMIT 30', db)
    multi_step = db.execute('SELECT event_id,n_timestamps FROM events WHERE n_timestamps > 1 ORDER BY max_score DESC LIMIT 1').fetchone()
event_columns = ['event_id', 'n_timestamps', 'duration_hours', 'cell_observations', 'max_score']
if 'mean_uncertainty' in events.columns:
    event_columns.append('mean_uncertainty')
display(events[event_columns])
EVENT_ID = int(multi_step[0]) if multi_step else (int(events.event_id.iloc[0]) if len(events) else None)
print('Evento selezionato:', EVENT_ID, '| timestamp:', multi_step[1] if multi_step else (int(events.n_timestamps.iloc[0]) if len(events) else 0))
print('Puoi sostituire EVENT_ID con un valore della tabella e rieseguire le celle successive.')

## Un timestamp
La prima figura mostra l'intera griglia geografica al timestamp di picco di `EVENT_ID`: il contorno azzurro indica le celle dell'evento e il riquadro tratteggiato delimita l'ingrandimento successivo. La seconda figura ingrandisce la stessa zona e lo stesso timestamp; gli altri score presenti nella zona restano visibili come contesto, mentre il contorno azzurro indica `EVENT_ID`. La scala colori della mappa completa si adatta all'intero timestamp; i pannelli ingranditi hanno ciascuno la propria barra colori. La tabella sotto le mappe elenca solo i cluster di `EVENT_ID` a quel timestamp. I pannelli confrontano score medio MC, threshold, opening, closing, cluster e deviazione standard MC (se disponibile); non mostrano da soli l'evoluzione temporale. Solo lo score medio determina la soglia e i cluster: la deviazione standard e diagnostica. Gli ID dei cluster sono globali, mentre `EVENT_ID` identifica l'evento temporale. I bordi esterni e le celle senza dati restano sfondo nella morfologia.

In [ ]:
with closing(sqlite3.connect(EVENT_DIR / 'events.sqlite')) as db:
    peak = db.execute('SELECT time_index FROM clusters WHERE event_id=? ORDER BY max_score DESC LIMIT 1', (EVENT_ID,)).fetchone() if EVENT_ID is not None else None
TIME_INDEX = int(peak[0]) if peak else min(4, metadata['shape'][0] - 1)
FRAME_EVENT_ID = EVENT_ID
FRAME_EVENT_DIR = Path(EVENT_DIR).resolve()
if EVENT_ID is not None:
    overview = plot_event_overview(EVENT_DIR, TIME_INDEX, EVENT_ID)
    display(overview)
    plt.close(overview)
figure, clusters = plot_frame(EVENT_DIR, TIME_INDEX, event_id=EVENT_ID) if EVENT_ID is not None else plot_frame(EVENT_DIR, TIME_INDEX)
display(figure)
plt.close(figure)
print('Evento selezionato:', EVENT_ID, '| timestamp di picco:', TIME_INDEX)
row = activity.iloc[TIME_INDEX]
print(f"In questo timestamp: {row.n_above_threshold}/{row.n_valid} localita' sopra soglia ({row.percent_above_threshold:.2f}%); {row.n_event_cells} celle nei cluster finali")
display(clusters)

## Evoluzione di un evento
La traiettoria, gli snapshot e la sezione Un timestamp seguono tutti lo stesso `EVENT_ID`. La prima sezione mostra il picco; le colonne degli snapshot corrispondono al primo, a un timestamp intermedio e all'ultimo timestamp dell'evento. Tutte le righe degli snapshot usano la stessa area geografica: **prima riga** score di tutte le celle della zona, con contorno azzurro dell'evento; **seconda riga** solo celle di `EVENT_ID`, con la stessa scala colori della prima; **terza riga** deviazione standard MC solo nelle celle dell'evento, se disponibile. Le celle vuote nella seconda e terza riga sono nascoste: possono avere uno score visibile nella prima riga. I quadrati sono celle della griglia a 0,5 gradi; la forma dell'evento deriva anche da threshold, opening e closing. Una deviazione MC alta indica variabilita tra i draw dropout, non una probabilita di anomalia. Se l'evento dura un solo timestamp, vedrai punti ma nessuna evoluzione temporale: scegline uno con `n_timestamps > 1`. Se un evento si divide, `event_steps` aggrega i rami nello stesso timestamp; `clusters` e `links` conservano i singoli rami e le relazioni split/merge. La durata include l'intervallo dell'ultimo campione; `elapsed_hours` riporta invece `end - start`.

In [ ]:
if EVENT_ID is not None:
    assert EVENT_ID == FRAME_EVENT_ID and Path(EVENT_DIR).resolve() == FRAME_EVENT_DIR, 'EVENT_ID o EVENT_DIR cambiato: riesegui prima la cella Un timestamp'
    figure, trajectory = plot_event(EVENT_DIR, EVENT_ID)
    print('Timestamp di questo evento:', len(trajectory))
    display(trajectory)
    snapshots = plot_event_snapshots(EVENT_DIR, EVENT_ID)
    with closing(sqlite3.connect(EVENT_DIR / 'events.sqlite')) as db:
        links = pd.read_sql_query(
            'SELECT l.* FROM links l JOIN clusters c ON c.cluster_id=l.source_cluster WHERE c.event_id=? LIMIT 200',
            db, params=(EVENT_ID,))
    display(links)
else:
    print('Nessun evento con questa configurazione.')

## Controllo della stagionalita'

Report annuale (stesso formato di prima) per lo z e, se `ORIGINAL_EVENT_DIR` esiste, per lo score originale, con una tabella di confronto. Se la normalizzazione funziona, la correlazione fra anni scende verso il nullo e i lift per mese e ora si avvicinano a 1. Ora il pannello degli eventi e' calcolato sugli eventi z. I report vanno in `EVENT_DIR/annual_report` (con il confronto in fondo a `summary.txt`) e `ORIGINAL_EVENT_DIR/annual_report`.

In [ ]:
def event_starts(directory):
    with closing(sqlite3.connect(Path(directory) / 'events.sqlite')) as db:
        return pd.read_sql_query('SELECT start FROM events', db).start


z_report = annual_report(activity, EVENT_DIR / 'annual_report', event_starts=event_starts(EVENT_DIR), header=[
    f'EVENT_DIR: {EVENT_DIR.name}', 'Score: z-score leave-one-year-out per localita x mese x ora UTC',
    f"Soglia: percentile {100 - metadata['config']['top_percent']:g} degli z = {activity.threshold.iloc[0]:.6g}"])
print(z_report['recurrence_line'])
for figure in z_report['figures']:
    display(figure)
    plt.close(figure)
if complete(ORIGINAL_EVENT_DIR):
    original_config = json.loads((ORIGINAL_EVENT_DIR / 'metadata.json').read_text(encoding='utf-8'))['config']
    original_activity = anomaly_activity(ORIGINAL_EVENT_DIR)
    original_report = annual_report(original_activity, ORIGINAL_EVENT_DIR / 'annual_report',
                                    event_starts=event_starts(ORIGINAL_EVENT_DIR), header=[
        f'EVENT_DIR: {ORIGINAL_EVENT_DIR.name}', 'Score: STGAN originale',
        f"Soglia: percentile {100 - original_config['top_percent']:g} = {original_activity.threshold.iloc[0]:.6g}"])
    for figure in original_report['figures']:
        plt.close(figure)
    comparison = pd.DataFrame({'score originale': seasonality_metrics(original_report),
                               'z climatologico': seasonality_metrics(z_report)})
    comparison.loc['eventi'] = [len(event_starts(ORIGINAL_EVENT_DIR)), len(event_starts(EVENT_DIR))]
    display(comparison.round(3))
    with open(z_report['report_dir'] / 'summary.txt', 'a', encoding='utf-8') as summary:
        summary.write('\n== Confronto con lo score originale ==\n' + comparison.round(4).to_string() + '\n')
    comparison.round(5).to_csv(z_report['report_dir'] / 'confronto.csv')
display(z_report['tables']['Tasso per mese'].round(3))
print(f"Report salvato in {z_report['report_dir']}: manda summary.txt (+ anno_settimana.png e lift.png)")